### Testing RAG Applications 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [1]:
#!pip install -qU langchain-chroma

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
# from langchain_ollama import ChatOllama

c:\Users\Admin\Documents\GEN_AI\LLM_Evaluation\Test_AI\Dev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


### If you are using Ollama Model within RAG then perform below step

In [ ]:
# llm = ChatOllama(
#     base_url="http://localhost:11434",
#     model = "qwen2.5:latest",
#     temperature=0.5,
#     max_tokens = 250
# )

### If you are using Groq API Model within RAG then perform below step

In [4]:
# Initialize Groq LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

In [5]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
# embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    # docs = retriever.get_relevant_documents(question)
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


C:\Users\Admin\AppData\Local\Temp\ipykernel_25380\3881388503.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


#### Output of the LLM Application

In [6]:
response = chain.invoke("What is MCP")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [7]:
print(response)

MCP stands for Model Context Protocol, a client-server architecture that provides a universal way for AI applications to interact with external systems by standardizing context.


### Testing RAG Application with RAGAs


In [8]:
test_data = [
    {
        "input": "What is MCP",

        "reference": "The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps."
    },
    {
        "input": "What is Relationship between function calling & Model Context Protocol",

        "reference": "The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI applications to context while leveraging function calling to make API interactions more consistent across different applications and model vendors."
    },
    {
        "input": "What are the core components of MCP, just give the heading",
        
        "reference":""" 
                    - MCP Client
                    - MCP Servers
                    - Protocol Handshake
                    - Capability Discovery
                """
    }
]

In [9]:
### Creating empty daaset to test cases

dataset = []

In [10]:
for question in test_data:
    context = retrieve_and_format(question["input"])   ### Context from the retriever
    response = chain.invoke(question["input"])         ### LLM Response
    dataset.append({
        "user_input": question['input'],
        "retrieved_contexts": [context],
        "response": response,
        "reference": question['reference']
    })

In [11]:
dataset

[{'user_input': 'What is MCP',
  'retrieved_contexts': ['affair. This dramatically lowers the barrier for developer adoption, especially among those already using AI-enabled tools. However, consumer-facing applications like Claude Desktop still require manual configuration with JSON files, highlighting an increasingly apparent gap between developer tooling and consumer use cases.Examples of MCP serversThe MCP ecosystem comprises a diverse range of servers including reference servers (created by the protocol maintainers as implementation examples),\n\npreventing accidental (or malicious) access to sensitive data stores.\xa0MCP client & server ecosystemSince its introduction in late 2024, MCP has experienced explosive growth. Some MCP marketplaces claim nearly 16,000 unique servers at the time of writing, but the real number (including those that aren’t made public) could be considerably higher.  Examples of MCP clientsThe MCP client ecosystem now includes:Claude Desktop: The original, f

In [14]:
from ragas.metrics import LLMContextRecall, NoiseSensitivity, Faithfulness, FactualCorrectness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas import (EvaluationDataset, evaluate)


evaluator_llm = LangchainLLMWrapper(llm)

evaluation_dataset = EvaluationDataset.from_list(dataset)

result = evaluate(dataset=evaluation_dataset, 
                  metrics=[LLMContextRecall(),
                           Faithfulness(),
                           AnswerRelevancy(),
                           FactualCorrectness()],
                  llm = evaluator_llm)

C:\Users\Admin\AppData\Local\Temp\ipykernel_25380\3486702305.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)
Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 12/12 [00:53<00:00,  4.49s/it]


In [15]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,answer_relevancy,factual_correctness(mode=f1)
0,What is MCP,[affair. This dramatically lowers the barrier ...,"MCP stands for Model Context Protocol, a unive...",The Model Context Protocol (MCP) addresses thi...,0.5,1.0,0.869156,0.60
1,What is Relationship between function calling ...,[more consistent. Relationship between functio...,The relationship between function calling and ...,The Model Context Protocol (MCP) builds on top...,1.0,1.0,0.923481,0.67
2,"What are the core components of MCP, just give...",[systems by standardizing context.Fig: MCP gen...,Core components,\n - MCP Client\n ...,0.5,1.0,0.000000,0.00
